# SSL Autoencoder Embeddings + t-SNE

Train an SSL autoencoder using the MARVEL dual-branch encoder (`ResNet` for MFCC, `EfficientNet` for log-mel), on **merged train + test** data using only audio features (`mfcc`, `mel_spectrogram`).

Pipeline:
1. Load cleaned train/test parquet
2. Keep audio tensors + metadata
3. Train `MarvelSSLAutoencoder` for reconstruction
4. Extract + normalize embeddings
5. Visualize t-SNE:
   - colored by `diagnosis_parkinsons_ds`
   - colored by `voice_perception__voice_quality_perception`
   - longitudinal evolution lines for test patients


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

import torch
from torch.utils.data import Dataset, DataLoader

from sklearn.manifold import TSNE

from marvel_model import MarvelMaskedAutoencoder

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

DATA_DIR = Path('/home/srl/B2AI/LLM/data')
TRAIN_PATH = DATA_DIR / 'cleaned_non_longitudinal_parkinson.parquet'
TEST_PATH = DATA_DIR / 'cleaned_longitudinal_parkinson.parquet'

print('Train path:', TRAIN_PATH)
print('Test  path:', TEST_PATH)


Device: cuda
Train path: /home/srl/B2AI/LLM/data/cleaned_non_longitudinal_parkinson.parquet
Test  path: /home/srl/B2AI/LLM/data/cleaned_longitudinal_parkinson.parquet


In [2]:
def to_2d_float32(x):
    """Robustly convert a parquet cell (ndarray, list-of-lists, etc.) to 2D float32."""
    # Already a proper 2D ndarray
    if isinstance(x, np.ndarray) and x.ndim == 2:
        return x.astype(np.float32)
    # From parquet: x is typically a list whose elements may be lists/arrays
    # of potentially different lengths (ragged). Stack row-by-row.
    try:
        arr = np.array(x, dtype=np.float32)
        if arr.ndim == 2:
            return arr
    except (ValueError, TypeError):
        pass
    # Fallback: convert each row individually and stack
    rows = [np.asarray(row, dtype=np.float32).ravel() for row in x]
    return np.stack(rows)  # (C, T)


def pad_or_truncate_2d(mat, target_t):
    # mat shape: (C, T)
    t = mat.shape[1]
    if t == target_t:
        return mat
    if t > target_t:
        return mat[:, :target_t]
    pad = np.zeros((mat.shape[0], target_t - t), dtype=mat.dtype)
    return np.concatenate([mat, pad], axis=1)


# Fixed sizes for CNN input
MFCC_T = 512
MEL_T = 512

train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)

train_df = train_df.copy()
test_df = test_df.copy()
train_df['split'] = 'train'
test_df['split'] = 'test'

all_df = pd.concat([train_df, test_df], ignore_index=True)
print('Train shape:', train_df.shape)
print('Test shape :', test_df.shape)
print('All shape  :', all_df.shape)

# Keep all columns except the raw audio arrays as metadata
audio_cols = {'mfcc', 'mel_spectrogram'}
meta_cols = [c for c in all_df.columns if c not in audio_cols]
meta_df = all_df[meta_cols].copy()
print(f'Metadata columns ({len(meta_cols)}):', meta_cols[:10], '...')

mfcc_list = []
mel_list = []
for _, row in all_df.iterrows():
    mfcc = to_2d_float32(row['mfcc'])
    mel = to_2d_float32(row['mel_spectrogram'])

    # fixed T for batching
    mfcc = pad_or_truncate_2d(mfcc, MFCC_T)
    mel = pad_or_truncate_2d(mel, MEL_T)

    # per-recording z-norm (robust training)
    mfcc = (mfcc - mfcc.mean(axis=1, keepdims=True)) / (mfcc.std(axis=1, keepdims=True) + 1e-8)
    mel = (mel - mel.mean(axis=1, keepdims=True)) / (mel.std(axis=1, keepdims=True) + 1e-8)

    mfcc_list.append(mfcc)
    mel_list.append(mel)

X_mfcc = np.stack(mfcc_list).astype(np.float32)  # (N, C_mfcc, T)
X_mel = np.stack(mel_list).astype(np.float32)    # (N, C_mel, T)

print('MFCC tensor:', X_mfcc.shape)
print('Log-mel tensor:', X_mel.shape)


Train shape: (1876, 74)
Test shape : (700, 75)
All shape  : (2576, 75)
Metadata columns (73): ['participant_id', 'session_id', 'task_name', 'n_frames_mfcc', 'n_frames_mel', 'diagnosis_parkinsons_ds', 'children', 'cognition', 'edu_level', 'ethnicity'] ...
MFCC tensor: (2576, 180, 512)
Log-mel tensor: (2576, 60, 512)


In [ ]:
class AudioSSLDataset(Dataset):
    def __init__(self, x_mfcc, x_mel):
        # convert to (N, 1, H, W)
        self.x_mfcc = torch.tensor(x_mfcc[:, None, :, :], dtype=torch.float32)
        self.x_mel = torch.tensor(x_mel[:, None, :, :], dtype=torch.float32)

    def __len__(self):
        return self.x_mfcc.shape[0]

    def __getitem__(self, idx):
        return self.x_mfcc[idx], self.x_mel[idx]


dataset = AudioSSLDataset(X_mfcc, X_mel)
loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=0, drop_last=False)

# ── Masked CNN Autoencoder (MAE-style) ────────────────────────────
model = MarvelMaskedAutoencoder(
    shared_dim=512,
    channels=(64, 128, 256, 512),
    patch_size=(16, 16),
    mask_ratio=0.50,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

max_epochs = 40
patience = 10          # stop if no improvement for this many epochs
loss_hist = []
best_loss = float('inf')
best_epoch = 0
best_state = None      # keep best model weights

model.train()
for epoch in range(1, max_epochs + 1):
    running = 0.0
    n_batches = 0

    for xb_mfcc, xb_mel in loader:
        xb_mfcc = xb_mfcc.to(device)
        xb_mel = xb_mel.to(device)

        # Masking happens inside the model during training
        recon_mfcc, recon_mel, mask_mfcc, mask_mel = model(xb_mfcc, xb_mel)
        loss = model.reconstruction_loss(
            x_mfcc=xb_mfcc,
            x_spec=xb_mel,
            recon_mfcc=recon_mfcc,
            recon_spec=recon_mel,
            mask_mfcc=mask_mfcc,
            mask_spec=mask_mel,
            mfcc_weight=1.0,
            spec_weight=1.0,
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running += loss.item()
        n_batches += 1

    epoch_loss = running / max(n_batches, 1)
    loss_hist.append(epoch_loss)

    # ── Early stopping check ──────────────────────────────────────
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        best_epoch = epoch
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        marker = ' ★'
    else:
        marker = ''

    print(f'Epoch {epoch:03d} | Recon loss: {epoch_loss:.5f}{marker}')

    if epoch - best_epoch >= patience:
        print(f'\n⏹  Early stopping at epoch {epoch} '
              f'(no improvement for {patience} epochs, '
              f'best = {best_loss:.5f} at epoch {best_epoch})')
        break

# Restore best model weights
if best_state is not None:
    model.load_state_dict(best_state)
    model.to(device)
    print(f'✅  Restored best weights from epoch {best_epoch} (loss = {best_loss:.5f})')

plt.figure(figsize=(8, 4))
plt.plot(loss_hist)
plt.title('Masked CNN Autoencoder Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Reconstruction loss (masked)')
plt.grid(alpha=0.2)
plt.show()


Model parameters: 27,535,364
Epoch 001 | Recon loss: 0.84800 ★
Epoch 002 | Recon loss: 0.78872 ★
Epoch 003 | Recon loss: 0.76662 ★
Epoch 004 | Recon loss: 0.74968 ★
Epoch 005 | Recon loss: 0.73854 ★
Epoch 006 | Recon loss: 0.73414 ★
Epoch 007 | Recon loss: 0.72508 ★
Epoch 008 | Recon loss: 0.72134 ★
Epoch 009 | Recon loss: 0.71426 ★
Epoch 010 | Recon loss: 0.71013 ★
Epoch 011 | Recon loss: 0.70679 ★
Epoch 012 | Recon loss: 0.70519 ★
Epoch 013 | Recon loss: 0.69753 ★
Epoch 014 | Recon loss: 0.69645 ★
Epoch 015 | Recon loss: 0.69190 ★
Epoch 016 | Recon loss: 0.68799 ★
Epoch 017 | Recon loss: 0.68593 ★
Epoch 018 | Recon loss: 0.68122 ★


KeyboardInterrupt: 

In [ ]:
# Extract embeddings for all recordings
model.eval()
with torch.no_grad():
    x_mfcc_t = torch.tensor(X_mfcc[:, None, :, :], dtype=torch.float32, device=device)
    x_mel_t = torch.tensor(X_mel[:, None, :, :], dtype=torch.float32, device=device)
    _, _, z, _, _ = model(x_mfcc_t, x_mel_t, return_embedding=True)
    emb = z.cpu().numpy().astype(np.float32)

# L2-normalize embeddings
emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-8)
print('Embeddings shape:', emb.shape)

emb_df = meta_df.copy()
for j in range(emb.shape[1]):
    emb_df[f'emb_{j:03d}'] = emb[:, j]

# Save embeddings with metadata
out_path = DATA_DIR / 'ssl_embeddings_marvel_autoencoder.parquet'
emb_df.to_parquet(out_path, index=False)
print('Saved:', out_path)
print('Saved shape:', emb_df.shape)


In [ ]:
# t-SNE projection on embeddings
print('Running t-SNE...')
tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate='auto',
    init='pca',
    random_state=SEED,
)
emb_2d = tsne.fit_transform(emb)

plot_df = emb_df[['participant_id', 'session_id', 'split']].copy()
for c in ['task_name', 'diagnosis_parkinsons_ds', 'voice_perception__voice_quality_perception', 'session_order']:
    if c in emb_df.columns:
        plot_df[c] = emb_df[c]
plot_df['tsne_x'] = emb_2d[:, 0]
plot_df['tsne_y'] = emb_2d[:, 1]


# 2) Color by diagnosis_parkinsons_ds
if 'diagnosis_parkinsons_ds' in plot_df.columns:
    sev = pd.to_numeric(plot_df['diagnosis_parkinsons_ds'], errors='coerce')
    plt.figure(figsize=(8, 6))
    sc = plt.scatter(plot_df['tsne_x'], plot_df['tsne_y'], c=sev, s=8, alpha=0.7, cmap='plasma')
    plt.colorbar(sc, label='diagnosis_parkinsons_ds')
    plt.title('t-SNE colored by raw H&Y severity')
    plt.xlabel('t-SNE 1')
    plt.ylabel('t-SNE 2')
    plt.grid(alpha=0.2)
    plt.show()

# 3) Color by voice perception quality
if 'voice_perception__voice_quality_perception' in plot_df.columns:
    vq = pd.to_numeric(plot_df['voice_perception__voice_quality_perception'], errors='coerce')
    plt.figure(figsize=(8, 6))
    sc = plt.scatter(plot_df['tsne_x'], plot_df['tsne_y'], c=vq, s=8, alpha=0.7, cmap='viridis')
    plt.colorbar(sc, label='voice_perception__voice_quality_perception')
    plt.title('t-SNE colored by voice quality perception')
    plt.xlabel('t-SNE 1')
    plt.ylabel('t-SNE 2')
    plt.grid(alpha=0.2)
    plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  Longitudinal trajectories on t-SNE — one line per test patient
# ══════════════════════════════════════════════════════════════════

# ── 1. Get chronological session order from session.tsv ───────────
import glob

sess_tsv_pattern = '/home/srl/B2AI/physionet.org/files/b2ai-voice/3.0.0/*/session.tsv'
sess_files = glob.glob(sess_tsv_pattern)
if sess_files:
    sess_order_df = pd.concat([pd.read_csv(f, sep='\t') for f in sess_files],
                               ignore_index=True)
    # session.tsv has columns like: participant_id, session_id, ...
    # Sort by participant + session date/order to get chronological ordering
    print(f"session.tsv loaded: {len(sess_order_df)} rows, cols: {list(sess_order_df.columns[:6])}")
else:
    sess_order_df = None
    print("⚠ No session.tsv found — will use alphabetical session_id as fallback")

# ── 2. Filter test patients from plot_df ──────────────────────────
test_plot = plot_df[plot_df['split'] == 'test'].copy()
print(f"Test recordings in plot_df: {len(test_plot)}")
print(f"Test patients: {sorted(test_plot['participant_id'].unique())}")

# ── 3. Session-level centroids (mean t-SNE coords per session) ───
session_centroids = (
    test_plot
    .groupby(['participant_id', 'session_id'], as_index=False)
    .agg(tsne_x=('tsne_x', 'mean'), tsne_y=('tsne_y', 'mean'))
)

# ── 4. Assign chronological session order ─────────────────────────
if sess_order_df is not None and 'session_id' in sess_order_df.columns:
    # Use session.tsv row order as proxy for chronological order
    sess_rank = (sess_order_df
                 .drop_duplicates('session_id')
                 .reset_index(drop=True)
                 .reset_index()
                 .rename(columns={'index': '_global_rank'}))
    session_centroids = session_centroids.merge(
        sess_rank[['session_id', '_global_rank']], on='session_id', how='left')
    # Per-patient rank
    session_centroids = session_centroids.sort_values(['participant_id', '_global_rank'])
    session_centroids['session_order'] = (
        session_centroids.groupby('participant_id').cumcount() + 1)
    session_centroids.drop(columns='_global_rank', inplace=True)
else:
    # Fallback: alphabetical
    session_centroids = session_centroids.sort_values(['participant_id', 'session_id'])
    session_centroids['session_order'] = (
        session_centroids.groupby('participant_id').cumcount() + 1)

session_centroids = session_centroids.sort_values(['participant_id', 'session_order'])
print(f"\nSession centroids ({len(session_centroids)} rows):")
for pid, grp in session_centroids.groupby('participant_id'):
    print(f"  {pid[:10]}…  {len(grp)} sessions")

# ── 5. Plot ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))

# Light background: train points
train_plot = plot_df[plot_df['split'] == 'train']
ax.scatter(train_plot['tsne_x'], train_plot['tsne_y'],
           s=5, alpha=0.12, c='lightgray', label='train (recordings)', zorder=1)

# One color per test patient
patients = sorted(session_centroids['participant_id'].unique())
cmap = plt.cm.tab10

for i, pid in enumerate(patients):
    grp = session_centroids[session_centroids['participant_id'] == pid].sort_values('session_order')
    color = cmap(i)
    short_id = pid[:8] + '…'

    # Trajectory line with markers
    ax.plot(grp['tsne_x'].values, grp['tsne_y'].values,
            '-o', color=color, linewidth=2.5, markersize=9,
            markeredgecolor='k', markeredgewidth=0.6,
            alpha=0.9, label=short_id, zorder=3)

    # Session number labels
    for _, row in grp.iterrows():
        ax.annotate(f"S{int(row['session_order'])}",
                    (row['tsne_x'], row['tsne_y']),
                    textcoords='offset points', xytext=(7, 7),
                    fontsize=8, color=color, fontweight='bold')

    # Start (open circle) and end (✕)
    ax.scatter(grp['tsne_x'].iloc[0], grp['tsne_y'].iloc[0],
               marker='o', s=140, facecolors='none', edgecolors=color,
               linewidths=2.5, zorder=4)
    ax.scatter(grp['tsne_x'].iloc[-1], grp['tsne_y'].iloc[-1],
               marker='X', s=110, c=[color], edgecolors='k',
               linewidths=0.5, zorder=4)

ax.set_xlabel('t-SNE 1', fontsize=12)
ax.set_ylabel('t-SNE 2', fontsize=12)
ax.set_title('Longitudinal trajectories on t-SNE — test patients\n'
             '(session centroids, ○ = first session, ✕ = last session)',
             fontsize=13, fontweight='bold')
ax.legend(loc='best', fontsize=9, framealpha=0.9, title='Patient ID')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  PCA on SSL embeddings — which metadata drives the structure?
# ══════════════════════════════════════════════════════════════════
from sklearn.decomposition import PCA
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

# ── 1. Run PCA on the L2-normalised embeddings ───────────────────
n_components = 20
pca = PCA(n_components=n_components, random_state=SEED)
emb_pca = pca.fit_transform(emb)   # (N, 20)

print(f"PCA on {emb.shape[0]} embeddings ({emb.shape[1]}-D) → {n_components} components")
print(f"Explained variance (top 10): "
      f"{[f'{v:.1%}' for v in pca.explained_variance_ratio_[:10]]}")
print(f"Cumulative (top 10):         "
      f"{[f'{v:.1%}' for v in np.cumsum(pca.explained_variance_ratio_[:10])]}")

# ── 2. Scree plot ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(range(1, n_components + 1), pca.explained_variance_ratio_ * 100,
            color='steelblue', edgecolor='k', linewidth=0.4)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_title('Scree Plot')
axes[0].grid(axis='y', alpha=0.3)

axes[1].plot(range(1, n_components + 1),
             np.cumsum(pca.explained_variance_ratio_) * 100,
             'o-', color='steelblue', markersize=5)
axes[1].set_xlabel('Principal Component')
axes[1].set_ylabel('Cumulative Explained Variance (%)')
axes[1].set_title('Cumulative Explained Variance')
axes[1].axhline(80, ls='--', color='gray', lw=0.8, label='80%')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ── 3. Correlate metadata with top PCA components ────────────────
# Select numeric metadata columns with some variance (nunique > 1)
SKIP = {'participant_id', 'session_id', 'task_name', 'split',
        'mfcc', 'mel_spectrogram'}
meta_numeric = []
for c in meta_df.columns:
    if c in SKIP:
        continue
    vals = pd.to_numeric(meta_df[c], errors='coerce')
    if vals.notna().sum() > 10 and vals.nunique() > 1:
        meta_numeric.append(c)

print(f"\nNumeric metadata columns with variance: {len(meta_numeric)}")

n_pcs = 10   # correlate with top 10 PCs
corr_matrix = np.zeros((len(meta_numeric), n_pcs))
pval_matrix = np.zeros_like(corr_matrix)

for i, col in enumerate(meta_numeric):
    vals = pd.to_numeric(meta_df[col], errors='coerce').values
    mask = ~np.isnan(vals)
    for j in range(n_pcs):
        if mask.sum() > 10:
            rho, pv = spearmanr(vals[mask], emb_pca[mask, j])
            corr_matrix[i, j] = rho
            pval_matrix[i, j] = pv

corr_df = pd.DataFrame(corr_matrix,
                        index=meta_numeric,
                        columns=[f'PC{k+1}' for k in range(n_pcs)])

# ── 4. Rank features by max absolute correlation across PCs ──────
corr_df['max_|ρ|'] = corr_df.abs().max(axis=1)
corr_df = corr_df.sort_values('max_|ρ|', ascending=False)

print("\nTop 20 metadata features by max |Spearman ρ| with any PC:")
display(corr_df.head(20).style.format("{:+.3f}").background_gradient(
    cmap='RdBu_r', vmin=-0.5, vmax=0.5,
    subset=[f'PC{k+1}' for k in range(n_pcs)]))

# ── 5. Heatmap of top 25 features vs top 10 PCs ─────────────────
top_n = 25
top_features = corr_df.index[:top_n]
plot_data = corr_df.loc[top_features, [f'PC{k+1}' for k in range(n_pcs)]]

fig, ax = plt.subplots(figsize=(10, max(8, top_n * 0.35)))
im = ax.imshow(plot_data.values, cmap='RdBu_r', aspect='auto',
               vmin=-0.5, vmax=0.5)

ax.set_xticks(range(n_pcs))
ax.set_xticklabels(plot_data.columns, fontsize=10)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features, fontsize=9)

# Annotate cells with correlation values
for i in range(len(top_features)):
    for j in range(n_pcs):
        val = plot_data.values[i, j]
        color = 'white' if abs(val) > 0.3 else 'black'
        ax.text(j, i, f'{val:+.2f}', ha='center', va='center',
                fontsize=7, color=color)

plt.colorbar(im, ax=ax, label='Spearman ρ', shrink=0.8)
ax.set_title(f'Top {top_n} metadata features × top {n_pcs} PCA components\n'
             '(Spearman correlation with SSL embeddings)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# ── 6. PCA scatter colored by top metadata features ──────────────
top4 = list(corr_df.index[:4])
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for idx, col in enumerate(top4):
    ax = axes[idx // 2][idx % 2]
    vals = pd.to_numeric(meta_df[col], errors='coerce').values

    sc = ax.scatter(emb_pca[:, 0], emb_pca[:, 1],
                    c=vals, s=6, alpha=0.6, cmap='plasma')
    plt.colorbar(sc, ax=ax, label=col, shrink=0.85)
    ax.set_xlabel('PC1', fontsize=10)
    ax.set_ylabel('PC2', fontsize=10)
    ax.set_title(f'PCA colored by {col}\n(max |ρ| = {corr_df.loc[col, "max_|ρ|"]:.3f})',
                 fontsize=10, fontweight='bold')
    ax.grid(alpha=0.2)

fig.suptitle('PCA projection colored by top metadata features',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  Reconstruction comparison — original vs decoded (MFCC & Mel)
# ══════════════════════════════════════════════════════════════════
import torch.nn.functional as F
import matplotlib.pyplot as plt

# ── Pick a few diverse samples to visualize ───────────────────────
# Choose from both train and test, spread across indices
n_samples = 6
total = X_mfcc.shape[0]
sample_idxs = np.linspace(0, total - 1, n_samples, dtype=int)

# ── Run model in eval mode to get reconstructions ─────────────────
model.eval()
with torch.no_grad():
    x_mfcc_batch = torch.tensor(
        X_mfcc[sample_idxs][:, None, :, :], dtype=torch.float32, device=device)
    x_mel_batch = torch.tensor(
        X_mel[sample_idxs][:, None, :, :], dtype=torch.float32, device=device)

    recon_mfcc, recon_mel, _, _ = model(x_mfcc_batch, x_mel_batch)

    # Move to CPU numpy
    orig_mfcc = x_mfcc_batch[:, 0].cpu().numpy()   # (n, 180, 512)
    orig_mel  = x_mel_batch[:, 0].cpu().numpy()     # (n, 60, 512)
    rec_mfcc  = recon_mfcc[:, 0].cpu().numpy()
    rec_mel   = recon_mel[:, 0].cpu().numpy()

# ── Helper: get sample label ──────────────────────────────────────
def _sample_label(idx):
    split = meta_df.iloc[idx]['split']
    pid = str(meta_df.iloc[idx]['participant_id'])[:6]
    sev = meta_df.iloc[idx].get('diagnosis_parkinsons_ds', '?')
    return f"#{idx} ({split}) P={pid}… sev={sev}"

# ══════════════════════════════════════════════════════════════════
#  Plot 1: MFCC — Original vs Reconstructed
# ══════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(n_samples, 3, figsize=(18, 3.2 * n_samples))

for i, idx in enumerate(sample_idxs):
    # Original
    im0 = axes[i, 0].imshow(orig_mfcc[i], aspect='auto', origin='lower', cmap='magma')
    axes[i, 0].set_title(f'Original MFCC\n{_sample_label(idx)}', fontsize=9)
    axes[i, 0].set_ylabel('Coefficient', fontsize=8)

    # Reconstructed
    im1 = axes[i, 1].imshow(rec_mfcc[i], aspect='auto', origin='lower', cmap='magma')
    axes[i, 1].set_title('Reconstructed MFCC', fontsize=9)

    # Difference
    diff = orig_mfcc[i] - rec_mfcc[i]
    vmax_d = max(abs(diff.min()), abs(diff.max()), 0.01)
    im2 = axes[i, 2].imshow(diff, aspect='auto', origin='lower',
                              cmap='RdBu_r', vmin=-vmax_d, vmax=vmax_d)
    mse = np.mean(diff ** 2)
    axes[i, 2].set_title(f'Difference (MSE={mse:.4f})', fontsize=9)
    plt.colorbar(im2, ax=axes[i, 2], fraction=0.03, pad=0.02)

    for j in range(3):
        axes[i, j].set_xlabel('Time frame', fontsize=8)

fig.suptitle('MFCC Reconstruction: Original vs Decoded',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════
#  Plot 2: Mel Spectrogram — Original vs Reconstructed
# ══════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(n_samples, 3, figsize=(18, 3.2 * n_samples))

for i, idx in enumerate(sample_idxs):
    # Original
    im0 = axes[i, 0].imshow(orig_mel[i], aspect='auto', origin='lower', cmap='magma')
    axes[i, 0].set_title(f'Original Mel Spectrogram\n{_sample_label(idx)}', fontsize=9)
    axes[i, 0].set_ylabel('Mel bin', fontsize=8)

    # Reconstructed
    im1 = axes[i, 1].imshow(rec_mel[i], aspect='auto', origin='lower', cmap='magma')
    axes[i, 1].set_title('Reconstructed Mel Spectrogram', fontsize=9)

    # Difference
    diff = orig_mel[i] - rec_mel[i]
    vmax_d = max(abs(diff.min()), abs(diff.max()), 0.01)
    im2 = axes[i, 2].imshow(diff, aspect='auto', origin='lower',
                              cmap='RdBu_r', vmin=-vmax_d, vmax=vmax_d)
    mse = np.mean(diff ** 2)
    axes[i, 2].set_title(f'Difference (MSE={mse:.4f})', fontsize=9)
    plt.colorbar(im2, ax=axes[i, 2], fraction=0.03, pad=0.02)

    for j in range(3):
        axes[i, j].set_xlabel('Time frame', fontsize=8)

fig.suptitle('Mel Spectrogram Reconstruction: Original vs Decoded',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Summary stats across ALL data ────────────────────────────────
print("\n── Reconstruction quality (full dataset) ──")
with torch.no_grad():
    # Process in batches to avoid OOM
    all_mfcc_mse, all_mel_mse = [], []
    bs = 64
    for start in range(0, total, bs):
        end = min(start + bs, total)
        xm = torch.tensor(X_mfcc[start:end, None], dtype=torch.float32, device=device)
        xs = torch.tensor(X_mel[start:end, None], dtype=torch.float32, device=device)
        rm, rs, _, _ = model(xm, xs)
        all_mfcc_mse.append(F.mse_loss(rm, xm, reduction='none').mean(dim=(1,2,3)).cpu().numpy())
        all_mel_mse.append(F.mse_loss(rs, xs, reduction='none').mean(dim=(1,2,3)).cpu().numpy())

    mfcc_mses = np.concatenate(all_mfcc_mse)
    mel_mses = np.concatenate(all_mel_mse)

print(f"  MFCC MSE:  mean={mfcc_mses.mean():.5f}  std={mfcc_mses.std():.5f}  "
      f"median={np.median(mfcc_mses):.5f}")
print(f"  Mel  MSE:  mean={mel_mses.mean():.5f}  std={mel_mses.std():.5f}  "
      f"median={np.median(mel_mses):.5f}")

# ── Distribution of per-sample MSE ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, mses, name in [(axes[0], mfcc_mses, 'MFCC'), (axes[1], mel_mses, 'Mel Spectrogram')]:
    ax.hist(mses, bins=50, color='steelblue', edgecolor='k', linewidth=0.4, alpha=0.8)
    ax.axvline(np.median(mses), color='red', ls='--', lw=1.5, label=f'median={np.median(mses):.4f}')
    ax.set_xlabel('Per-sample MSE')
    ax.set_ylabel('Count')
    ax.set_title(f'{name} Reconstruction MSE Distribution')
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()